# Chương 11 — Language Modeling: Thực hành N-gram

**Mục tiêu:** thực hành Unigram, Bigram, Trigram; xác suất MLE; Perplexity; Add-k smoothing; Backoff; Deleted Interpolation; và Class N-grams. Cuối notebook có phần tùy chọn dùng `NLinh_1.wav` để tạo văn bản bằng ASR và đếm N-gram.

> **Lưu ý về dữ liệu:** một file audio ngắn không phải là ngữ liệu đủ lớn để huấn luyện hoặc đánh giá mô hình ngôn ngữ. Notebook dùng một ngữ liệu tiếng Việt minh họa gồm nhiều câu để tính toán ổn định hơn; phần audio chỉ là bài tập kết nối quy trình tiếng nói với Language Modeling.

## 1. Ôn lại các công thức cốt lõi

### 1.1. MLE cho N-gram

Với lịch sử $h$ và từ tiếp theo $w$:

$$
P_{\text{MLE}}(w\mid h)=\frac{C(h,w)}{C(h)}
$$

- $C(h,w)$: số lần lịch sử $h$theo sau bởi từ $w\$
- $C(h)$ số lần lịch sử $h\$xuất hiện.

Bigram: $P(w_i\mid w_{i-1})=C(w_{i-1},w_i)/C(w_{i-1})$.

Trigram: $P(w_i\mid w_{i-2},w_{i-1})=C(w_{i-2},w_{i-1},w_i)/C(w_{i-2},w_{i-1})$.

### 1.2. Perplexity

$$
H = -\frac{1}{N}\sum_{i=1}^{N}\log_2 P(w_i \mid h_i)
$$
$$
PP = 2^{H}
$$


là số token được chấm điểm (thường không tính token bắt đầu câu). Perplexity thấp hơn thường tốt hơn **khi so sánh trên cùng tập kiểm tra và cùng quy ước token hóa**. Nếu mô hình gán xác suất 0 cho một token, perplexity trở thành vô hạn; đây là lý do cần smoothing/backoff. 


### 1.3. Add-k smoothing

$$
P_{\mathrm{add-k}}(w \mid h) = \frac{C(h,w)+k}{C(h)+kV}
$$

$k>0$, $V$ là kích thước từ vựng đầu ra. Trong notebook, ta dùng add-one ($k=1$) làm ví dụ đối chiếu.

## 2. Chuẩn bị ngữ liệu tiếng Việt

Ngữ liệu dưới đây là **dữ liệu minh họa được viết sẵn**, không phải bản chép lời của `NLinh_1.wav`. Ta chia câu thành tập train và test để tránh đánh giá trên chính dữ liệu đã đếm. Token hóa bằng khoảng trắng nhằm giữ bài thực hành đơn giản; với tiếng Việt, cách này chưa tách đầy đủ các từ đa âm tiết theo nghĩa ngôn ngữ học.

In [2]:
import math
import random
from collections import Counter, defaultdict

random.seed(7)

sentences = [
    "mô hình ngôn ngữ dự đoán từ tiếp theo",
    "mô hình n gram dùng ngữ cảnh để dự đoán từ",
    "ngôn ngữ tự nhiên có nhiều cách diễn đạt",
    "mô hình bigram xét một từ trước đó",
    "mô hình trigram xét hai từ trước đó",
    "xác suất n gram được tính từ tần suất",
    "smoothing giúp xử lý n gram chưa xuất hiện",
    "perplexity đo mức độ khó đoán của văn bản",
    "mô hình tốt thường có perplexity thấp hơn",
    "ngữ liệu huấn luyện cần có nhiều câu",
    "ngữ liệu kiểm tra dùng để đánh giá mô hình",
    "backoff chuyển về mô hình bậc thấp hơn",
    "deleted interpolation kết hợp nhiều bậc n gram",
    "class n gram gom từ thành các lớp",
    "mô hình ngôn ngữ có thể dùng cho nhận dạng tiếng nói",
    "nhận dạng tiếng nói chuyển âm thanh thành văn bản",
    "văn bản được tách thành các token",
    "xác suất câu là tích xác suất có điều kiện",
    "từ hiếm gây khó khăn cho mô hình ngôn ngữ",
    "mô hình unigram không dùng ngữ cảnh trước"
]

# Chia tập theo câu; với bài thực hành nhỏ, đây là ví dụ minh họa.
train_sentences = sentences[:15]
test_sentences = sentences[15:]

print("Số câu train:", len(train_sentences))
print("Số câu test :", len(test_sentences))
print("\nVí dụ train:", train_sentences[0])
print("Ví dụ test :", test_sentences[0])

Số câu train: 15
Số câu test : 5

Ví dụ train: mô hình ngôn ngữ dự đoán từ tiếp theo
Ví dụ test : nhận dạng tiếng nói chuyển âm thanh thành văn bản


## 3. Đếm Unigram, Bigram, Trigram

Dùng `<s>` để đánh dấu đầu câu và `</s>` để đánh dấu cuối câu. Khi tính perplexity, ta sẽ bỏ qua `<s>` vì đây là ngữ cảnh chứ không phải từ cần dự đoán.

In [3]:
BOS = "<s>"
EOS = "</s>"
UNK = "<unk>"

def tokenize_sentence(sentence):
    return sentence.lower().strip().split()

def add_boundaries(sentence):
    return [BOS] + tokenize_sentence(sentence) + [EOS]

def count_ngrams(corpus, n):
    counts = Counter()
    for sentence in corpus:
        tokens = add_boundaries(sentence)
        for i in range(len(tokens) - n + 1):
            counts[tuple(tokens[i:i+n])] += 1
    return counts

unigram_counts = count_ngrams(train_sentences, 1)
bigram_counts = count_ngrams(train_sentences, 2)
trigram_counts = count_ngrams(train_sentences, 3)

print("10 Unigram phổ biến:")
for gram, count in unigram_counts.most_common(10):
    print(f"{gram[0]:20s} {count}")

print("\n10 Bigram phổ biến:")
for gram, count in bigram_counts.most_common(10):
    print(f"{gram} -> {count}")

print("\n10 Trigram phổ biến:")
for gram, count in trigram_counts.most_common(10):
    print(f"{gram} -> {count}")

10 Unigram phổ biến:
<s>                  15
</s>                 15
mô                   8
hình                 8
ngữ                  6
từ                   6
n                    5
gram                 5
có                   4
ngôn                 3

10 Bigram phổ biến:
('mô', 'hình') -> 8
('<s>', 'mô') -> 6
('n', 'gram') -> 5
('ngôn', 'ngữ') -> 3
('hình', 'ngôn') -> 2
('dự', 'đoán') -> 2
('đoán', 'từ') -> 2
('có', 'nhiều') -> 2
('từ', 'trước') -> 2
('trước', 'đó') -> 2

10 Trigram phổ biến:
('<s>', 'mô', 'hình') -> 6
('mô', 'hình', 'ngôn') -> 2
('hình', 'ngôn', 'ngữ') -> 2
('dự', 'đoán', 'từ') -> 2
('từ', 'trước', 'đó') -> 2
('trước', 'đó', '</s>') -> 2
('thấp', 'hơn', '</s>') -> 2
('<s>', 'ngữ', 'liệu') -> 2
('ngôn', 'ngữ', 'dự') -> 1
('ngữ', 'dự', 'đoán') -> 1


### Tự kiểm tra bằng một Bigram

Nếu cụm `mô hình` xuất hiện 4 lần và `mô hình ngôn` xuất hiện 2 lần trong tập train, thì:

$$
P(\text{ngôn}\mid\text{mô hình})=\frac{C(\text{mô hình ngôn})}{C(\text{mô hình})}=\frac{2}{4}=0.5
$$

Các con số thực tế sẽ được lấy từ ngữ liệu chạy bên dưới, không cần đoán trước.

In [4]:
def context_count(corpus, context):
    # Đếm số lần context xuất hiện như lịch sử trong N-gram.
    n = len(context) + 1
    grams = count_ngrams(corpus, n)
    return sum(c for gram, c in grams.items() if gram[:-1] == tuple(context))

def mle_probability(corpus, history, word):
    n = len(history) + 1
    counts = count_ngrams(corpus, n)
    numerator = counts[tuple(history) + (word,)]
    denominator = context_count(corpus, history)
    return numerator / denominator if denominator else 0.0

history = ("mô", "hình")
word = "ngôn"
num = count_ngrams(train_sentences, 3)[history + (word,)]
den = context_count(train_sentences, history)
print("C(mô hình ngôn) =", num)
print("C(mô hình)      =", den)
print("P_MLE(ngôn | mô hình) =", num / den if den else 0.0)

C(mô hình ngôn) = 2
C(mô hình)      = 8
P_MLE(ngôn | mô hình) = 0.25


## 4. Tạo từ vựng và xử lý từ chưa biết (`<unk>`)

Ta xây từ vựng từ tập train. Những từ trong tập test nhưng chưa xuất hiện trong train được đổi thành `<unk>`. Đây là một cách xử lý cơ bản; hệ thống thực tế thường cần chiến lược từ vựng và chuẩn hóa kỹ hơn.

In [5]:
train_vocab = set()
for sentence in train_sentences:
    train_vocab.update(tokenize_sentence(sentence))
vocab = train_vocab | {BOS, EOS, UNK}

def map_oov(sentence):
    return [w if w in train_vocab else UNK for w in tokenize_sentence(sentence)]

print("Kích thước từ vựng train:", len(vocab))
print("Từ test chưa có trong train:")
for sentence in test_sentences:
    oov = [w for w in tokenize_sentence(sentence) if w not in train_vocab]
    if oov:
        print(sentence, "=>", oov)

Kích thước từ vựng train: 83
Từ test chưa có trong train:
nhận dạng tiếng nói chuyển âm thanh thành văn bản => ['âm', 'thanh']
văn bản được tách thành các token => ['tách', 'token']
xác suất câu là tích xác suất có điều kiện => ['là', 'tích', 'điều', 'kiện']
từ hiếm gây khó khăn cho mô hình ngôn ngữ => ['hiếm', 'gây', 'khăn']
mô hình unigram không dùng ngữ cảnh trước => ['unigram', 'không']


## 5. Xác suất Bigram: MLE và Add-one smoothing

MLE có thể gán xác suất 0 cho bigram chưa từng xuất hiện. Add-one cộng 1 vào mỗi khả năng trong từ vựng, giúp xác suất không bằng 0.

$$
P_{\text{add-1}}(w\mid h)=\frac{C(h,w)+1}{C(h)+V}
$$

Ở đây $V$ là số từ có thể được dự đoán. Ta dùng từ vựng train cộng `<unk>` và `</s>`; không xem `<s>` là từ đầu ra.

In [6]:
OUTPUT_VOCAB = (train_vocab - {BOS}) | {EOS, UNK}
V = len(OUTPUT_VOCAB)

def bigram_mle(history, word):
    return mle_probability(train_sentences, (history,), word)

def bigram_add_one(history, word):
    c_hw = bigram_counts[(history, word)]
    c_h = sum(c for (h, w), c in bigram_counts.items() if h == history)
    return (c_hw + 1) / (c_h + V)

examples = [
    ("mô", "hình"),
    ("mô", "khôngcótrongtrain"),
    ("ngôn", "ngữ"),
]
for h, w in examples:
    p_mle = bigram_mle(h, w)
    p_add1 = bigram_add_one(h, w)
    print(f"P({w}|{h}): MLE={p_mle:.4f}, Add-one={p_add1:.4f}")

P(hình|mô): MLE=1.0000, Add-one=0.1000
P(khôngcótrongtrain|mô): MLE=0.0000, Add-one=0.0111
P(ngữ|ngôn): MLE=1.0000, Add-one=0.0471


## 6. Backoff: khi N-gram bậc cao không có dữ liệu

**Ý tưởng:** thử xác suất trigram trước; nếu trigram chưa thấy hoặc không có xác suất, lùi về bigram; nếu bigram cũng không có thì lùi về unigram.

Đây là **backoff đơn giản để minh họa**. Backoff thực tế như Katz backoff có hệ số giảm (discount) và trọng số backoff được ước lượng để phân phối xác suất được chuẩn hóa tốt hơn. Vì vậy, không nên gọi đoạn mã dưới đây là triển khai đầy đủ Katz backoff.

In [7]:
def unigram_probability(word):
    # MLE unigram trên token dự đoán, với smoothing add-one để tránh xác suất 0.
    predicted_counts = Counter()
    for sentence in train_sentences:
        predicted_counts.update(tokenize_sentence(sentence) + [EOS])
    return (predicted_counts[word] + 1) / (sum(predicted_counts.values()) + V)

def bigram_probability_backoff(history, word):
    # history là tuple có thể dài 0, 1 hoặc 2 token
    if len(history) >= 2:
        p = mle_probability(train_sentences, history[-2:], word)
        if p > 0:
            return p, "trigram MLE"
        history = history[-1:]
    if len(history) == 1:
        p = mle_probability(train_sentences, history, word)
        if p > 0:
            return p, "bigram MLE"
    return unigram_probability(word), "unigram add-one fallback"

for hist, target in [
    (("mô", "hình"), "ngôn"),
    (("mô", "hình"), "nhận"),
    (("khôngcó", "ngữcảnh"), "ngôn"),
]:
    p, source = bigram_probability_backoff(hist, target)
    print(f"history={hist}, target={target:10s} -> P={p:.6f} ({source})")

history=('mô', 'hình'), target=ngôn       -> P=0.250000 (trigram MLE)
history=('mô', 'hình'), target=nhận       -> P=0.008658 (unigram add-one fallback)
history=('khôngcó', 'ngữcảnh'), target=ngôn       -> P=0.017316 (unigram add-one fallback)


## 7. Deleted Interpolation

Deleted interpolation kết hợp xác suất từ nhiều bậc N-gram:

$$
\begin{aligned}
P(w_i\mid h_i) &= \lambda_3\, P_{\text{tri}}(w_i\mid w_{i-2},w_{i-1}) \\
&\quad + \lambda_2\, P_{\text{bi}}(w_i\mid w_{i-1}) \\
&\quad + \lambda_1\, P_{\text{uni}}(w_i)
\end{aligned}
$$

với $\lambda_1+\lambda_2+\lambda_3=1$, và mỗi $\lambda\ge0$

Trong thuật toán deleted interpolation chuẩn, các trọng số được ước lượng bằng cách xem mỗi sự kiện như thể nó bị xóa khỏi dữ liệu, so sánh các ước lượng bậc unigram/bigram/trigram còn lại rồi cộng điểm cho bậc dự đoán tốt nhất. Đoạn mã sau là **bản thực hành minh họa cơ chế kết hợp**, dùng trọng số cố định để dễ quan sát; nó không tuyên bố đã ước lượng deleted-interpolation weights chuẩn từ corpus.

In [8]:
# Trọng số minh họa; tổng bằng 1.
LAMBDA_1 = 0.2
LAMBDA_2 = 0.3
LAMBDA_3 = 0.5
assert abs(LAMBDA_1 + LAMBDA_2 + LAMBDA_3 - 1.0) < 1e-9

def unigram_add_one(word):
    return unigram_probability(word)

def bigram_add_one_prob(history_word, word):
    c_hw = bigram_counts[(history_word, word)]
    c_h = sum(c for (h, w), c in bigram_counts.items() if h == history_word)
    return (c_hw + 1) / (c_h + V)

def trigram_add_one_prob(history2, word):
    c_hww = trigram_counts[(history2[0], history2[1], word)]
    c_h = sum(c for (a, b, w), c in trigram_counts.items()
              if (a, b) == tuple(history2))
    return (c_hww + 1) / (c_h + V)

def deleted_interpolation_demo(history2, word):
    p1 = unigram_add_one(word)
    p2 = bigram_add_one_prob(history2[-1], word)
    p3 = trigram_add_one_prob(history2[-2:], word)
    p = LAMBDA_1*p1 + LAMBDA_2*p2 + LAMBDA_3*p3
    return p, {"unigram": p1, "bigram": p2, "trigram": p3}

p, components = deleted_interpolation_demo(("mô", "hình"), "ngôn")
print("Các xác suất thành phần:", components)
print("P_interpolated =", round(p, 6))
print("lambda:", LAMBDA_1, LAMBDA_2, LAMBDA_3)

Các xác suất thành phần: {'unigram': 0.017316017316017316, 'bigram': 0.03333333333333333, 'trigram': 0.03333333333333333}
P_interpolated = 0.03013
lambda: 0.2 0.3 0.5


## 8. Class N-grams

Class N-gram thay vì mô hình hóa từng từ riêng lẻ, ta ánh xạ từ vào lớp, ví dụ:

- `mô hình`, `ngôn ngữ`, `perplexity` → `CLASS_TECH`
- `dự đoán`, `đánh giá`, `xử lý` → `CLASS_ACTION`
- `văn bản`, `âm thanh`, `ngữ liệu` → `CLASS_DATA`

Ví dụ dưới đây dùng ánh xạ thủ công nhỏ để minh họa. Trong ứng dụng thực tế, các lớp có thể được thiết kế theo ngữ nghĩa/từ loại hoặc học từ dữ liệu. Lưu ý: gom lớp làm mất một phần thông tin từ vựng; class N-gram thường kết hợp xác suất lớp với xác suất từ trong lớp chứ không chỉ thay mọi từ bằng nhãn lớp.

In [9]:
CLASS_MAP = {
    "mô": "CLASS_TECH", "hình": "CLASS_TECH", "ngôn": "CLASS_TECH",
    "ngữ": "CLASS_TECH", "perplexity": "CLASS_TECH", "bigram": "CLASS_TECH",
    "trigram": "CLASS_TECH", "unigram": "CLASS_TECH", "n": "CLASS_TECH",
    "dự": "CLASS_ACTION", "đoán": "CLASS_ACTION", "đánh": "CLASS_ACTION",
    "giá": "CLASS_ACTION", "xử": "CLASS_ACTION", "lý": "CLASS_ACTION",
    "văn": "CLASS_DATA", "bản": "CLASS_DATA", "âm": "CLASS_DATA",
    "thanh": "CLASS_DATA", "ngữ": "CLASS_TECH", "liệu": "CLASS_DATA",
}

def word_to_class(word):
    if word in {BOS, EOS}:
        return word
    return CLASS_MAP.get(word, "CLASS_OTHER")

def class_sequence(sentence):
    return [word_to_class(w) for w in add_boundaries(sentence)]

class_train = [" ".join(class_sequence(s)) for s in train_sentences]
class_bigram_counts = count_ngrams(class_train, 2)

print("Câu gốc:")
print(train_sentences[0])
print("\nChuỗi lớp:")
print(class_sequence(train_sentences[0]))
print("\nCác class bigram phổ biến:")
for gram, count in class_bigram_counts.most_common(10):
    print(gram, "->", count)

Câu gốc:
mô hình ngôn ngữ dự đoán từ tiếp theo

Chuỗi lớp:
['<s>', 'CLASS_TECH', 'CLASS_TECH', 'CLASS_TECH', 'CLASS_TECH', 'CLASS_ACTION', 'CLASS_ACTION', 'CLASS_OTHER', 'CLASS_OTHER', 'CLASS_OTHER', '</s>']

Các class bigram phổ biến:
('class_other', 'class_other') -> 63
('class_tech', 'class_tech') -> 16
('<s>', '<s>') -> 15
('</s>', '</s>') -> 15
('class_tech', 'class_other') -> 14
('class_other', '</s>') -> 13
('<s>', 'class_tech') -> 10
('class_other', 'class_tech') -> 6
('<s>', 'class_other') -> 5
('class_action', 'class_action') -> 4


## 9. Tính log-probability và Perplexity trên tập test

Ta so sánh Bigram MLE, Bigram Add-one, và Trigram Add-one. MLE có thể gặp xác suất 0 nên ta dùng `-inf` cho log xác suất và báo Perplexity vô hạn nếu có sự kiện chưa thấy. Add-one cho phép tính được điểm số hữu hạn.

Để so sánh công bằng, các mô hình phải dùng **cùng tập test, cách token hóa và quy ước tính token**.

In [10]:
def score_sentence_bigram_mle(sentence):
    tokens = [BOS] + map_oov(sentence) + [EOS]
    logp = 0.0
    n_scored = 0
    for i in range(1, len(tokens)):
        p = mle_probability(train_sentences, (tokens[i-1],), tokens[i])
        if p <= 0:
            return float("-inf"), n_scored + 1
        logp += math.log2(p)
        n_scored += 1
    return logp, n_scored

def score_sentence_bigram_add1(sentence):
    tokens = [BOS] + map_oov(sentence) + [EOS]
    logp = 0.0
    n_scored = 0
    for i in range(1, len(tokens)):
        p = bigram_add_one(tokens[i-1], tokens[i])
        logp += math.log2(p)
        n_scored += 1
    return logp, n_scored

def score_sentence_trigram_add1(sentence):
    tokens = [BOS, BOS] + map_oov(sentence) + [EOS]
    logp = 0.0
    n_scored = 0
    for i in range(2, len(tokens)):
        history = (tokens[i-2], tokens[i-1])
        p = trigram_add_one_prob(history, tokens[i])
        logp += math.log2(p)
        n_scored += 1
    return logp, n_scored

def corpus_perplexity(scorer, corpus):
    total_logp = 0.0
    total_tokens = 0
    for sentence in corpus:
        logp, n = scorer(sentence)
        if logp == float("-inf"):
            return float("inf")
        total_logp += logp
        total_tokens += n
    return 2 ** (-total_logp / total_tokens)

for name, scorer in [
    ("Bigram MLE", score_sentence_bigram_mle),
    ("Bigram Add-one", score_sentence_bigram_add1),
    ("Trigram Add-one", score_sentence_trigram_add1),
]:
    print(f"{name:20s} Perplexity = {corpus_perplexity(scorer, test_sentences)}")

Bigram MLE           Perplexity = inf
Bigram Add-one       Perplexity = 60.38760552283535
Trigram Add-one      Perplexity = 71.06145695013473


### Cách đọc kết quả

- Nếu Bigram MLE gặp một bigram chưa từng xuất hiện trong train, nó có thể trả về Perplexity vô hạn.
- Add-one thường làm Perplexity hữu hạn, nhưng không đảm bảo luôn thấp hơn trên mọi tập dữ liệu.
- Trigram có ngữ cảnh dài hơn, nhưng cũng có nhiều N-gram hiếm hơn; smoothing và kích thước dữ liệu ảnh hưởng mạnh đến kết quả.
- Không kết luận mô hình nào “tốt nhất” chỉ từ ngữ liệu đồ chơi nhỏ này.

## 10. Tùy chọn: dùng `NLinh_1.wav` để tạo văn bản rồi đếm N-gram

Phần này nối bài Language Modeling với dữ liệu âm thanh. Trước hết, kiểm tra audio và phát nghe. Sau đó có thể dùng Whisper để tạo transcript. Whisper là ASR, không phải mô hình N-gram mà ta vừa xây dựng.

In [11]:
from pathlib import Path
audio_path = Path(r"D:\audio & speech\TH\TH_TrenLop\NLinh_1.wav")
print("Tìm thấy file:", audio_path.exists(), "|", audio_path)

try:
    import soundfile as sf
    audio, sr = sf.read(audio_path)
    print("Sample rate:", sr)
    print("Shape:", audio.shape)
    print("Thời lượng (giây):", len(audio) / sr)
except Exception as e:
    print("Chưa đọc được audio. Kiểm tra soundfile và đường dẫn:", repr(e))

Tìm thấy file: True | D:\audio & speech\TH\TH_TrenLop\NLinh_1.wav
Sample rate: 48000
Shape: (286080, 2)
Thời lượng (giây): 5.96


In [12]:
# Trong Jupyter, có thể phát audio nếu IPython hỗ trợ.
try:
    from IPython.display import Audio, display
    display(Audio(filename=str(audio_path)))
except Exception as e:
    print("Không phát được trực tiếp trong môi trường này:", repr(e))

### 10.1. Nhận dạng tiếng nói bằng Whisper (tùy chọn)

Cài trong môi trường notebook nếu cần:

```bash
pip install -U openai-whisper
```

Whisper thường cần FFmpeg được cài trên máy. Nếu không cài được ASR, có thể tự nghe và nhập transcript vào ô `transcript` bên dưới. Không nên tự bịa transcript vì mọi thống kê N-gram sau đó phụ thuộc vào văn bản chính xác.

In [18]:
!pip install -U openai-whisper

     ---------------------------------------- 0.0/803.2 kB ? eta -:--:--
     ---------------------------------------- 0.0/803.2 kB ? eta -:--:--
     ------------- -------------------------- 262.1/803.2 kB ? eta -:--:--
     ---------------------------------------- 803.2/803.2 kB 3.4 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/944.4 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/944.4 kB ? eta -:--:--
   ---------------------------------------- 944.4/944.4 kB 2.1 MB/s  0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=804013 sha256=3f96f20ccf20e097815b884bdc2efa27dba87dbade83


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import shutil
print(shutil.which("ffmpeg"))

C:\Users\pc\AppData\Local\Microsoft\WinGet\Links\ffmpeg.EXE


In [13]:
#transcript = ""  # Dán transcript tiếng Việt thực tế của NLinh_1.wav vào đây.

# Tùy chọn tự nhận dạng bằng Whisper:
import whisper
asr_model = whisper.load_model("base")
result = asr_model.transcribe(str(audio_path), language="Vietnamese")
transcript = result["text"]
print("Transcript:", transcript)

if transcript.strip():
    audio_tokens = tokenize_sentence(transcript)
    print("Số token:", len(audio_tokens))
    print("Tokens:", audio_tokens)
else:
    print("Chưa có transcript. Hãy chạy Whisper hoặc nhập bản chép lời đã kiểm tra.")

c:\Users\pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcript:  Họt 2, 3, 4, 5
Số token: 5
Tokens: ['họt', '2,', '3,', '4,', '5']


### 10.2. Đếm N-gram trên transcript audio

Đây là thống kê trên một câu/đoạn rất ngắn, chỉ nhằm minh họa cách đếm. Không dùng các tần suất này để khẳng định mô hình ngôn ngữ tốt hay kém.

In [14]:
if transcript.strip():
    audio_corpus = [transcript]
    audio_uni = count_ngrams(audio_corpus, 1)
    audio_bi = count_ngrams(audio_corpus, 2)
    audio_tri = count_ngrams(audio_corpus, 3)

    print("Unigram:")
    print(audio_uni)
    print("\nBigram:")
    print(audio_bi)
    print("\nTrigram:")
    print(audio_tri)
else:
    print("Bỏ qua vì chưa nhập transcript.")

Unigram:
Counter({('<s>',): 1, ('họt',): 1, ('2,',): 1, ('3,',): 1, ('4,',): 1, ('5',): 1, ('</s>',): 1})

Bigram:
Counter({('<s>', 'họt'): 1, ('họt', '2,'): 1, ('2,', '3,'): 1, ('3,', '4,'): 1, ('4,', '5'): 1, ('5', '</s>'): 1})

Trigram:
Counter({('<s>', 'họt', '2,'): 1, ('họt', '2,', '3,'): 1, ('2,', '3,', '4,'): 1, ('3,', '4,', '5'): 1, ('4,', '5', '</s>'): 1})


## 11. Bài tập tự luyện

1. Đổi `n=2` thành `n=3` trong `count_ngrams` và giải thích mẫu số khi tính xác suất trigram.
2. Tìm một bigram không xuất hiện trong train; so sánh MLE và Add-one.
3. Thay đổi trọng số $\lambda_1,\lambda_2,\lambda_3$ trong Deleted Interpolation demo, luôn bảo đảm tổng bằng 1. Quan sát xác suất thay đổi thế nào.
4. Thử thay cách ánh xạ `CLASS_MAP`; đếm lại class bigram.
5. Nhập transcript đã kiểm tra của `NLinh_1.wav` rồi in unigram, bigram, trigram. Giải thích vì sao corpus quá ngắn.
6. Muốn làm Deleted Interpolation đúng theo thuật toán chuẩn, cần triển khai bước leave-one-out/deleted estimation để ước lượng các trọng số từ dữ liệu, thay vì chọn tay.

## Kết luận

Notebook này tập trung vào các mục chính của bài thực hành Language Modeling: đếm N-gram, MLE, Perplexity, Add-one smoothing, backoff minh họa, deleted interpolation minh họa và class N-grams. Phần `NLinh_1.wav` chỉ là cầu nối từ audio sang transcript; mô hình ngôn ngữ được đánh giá trên ngữ liệu văn bản nhiều câu để tránh kết luận từ vài giây âm thanh.